In [7]:

import json
from sqlalchemy.orm import Session, sessionmaker
from pathlib import Path
from lib.db.database import engine
from lib.db.crud.authors.get import get_author
from lib.db.helpers.normalize_name import normalize_for_search
from lib.db.models import Author, PublicationContributor
from sqlalchemy import select
from lib.similarity.main import check_similarity
from lib.parser.author_crossref import parser_contributor
from lib.db.crud.authors.vinculate_publication import authors_to_publication
from lib.db.crud.authors.profile import get_or_create_profile
from lib.db.models import Affiliation
from lib.db.crud.authors.get_or_create import get_or_create_author
from lib.db.crud.authors.vinculate_affiliation import affiliation_to_author
from lib.parser.author_crossref import parser_author_crossref
from lib.parser.funders import parse_funder
from lib.parser.container import parser_container
from lib.parser.publication import parser_publication
from lib.db.crud.publication import get_or_create_publication
from lib.db.crud.container import get_or_create_container
from typing import List, Dict, Any, Optional
from lib.db.models import Publication
# from lib.db.models import Funder
# from lib.db.models import PublicationFunder
from lib.parser.references import parser_references
    
from lib.db.crud.funders.get_or_create import ingest_publication_funders
from lib.db.crud.references.replace_publication import replace_publication_references
from lib.parser.metrics import extract_metrics

import re
import unicodedata

In [4]:
SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)
session = SessionLocal()

In [3]:
artigos = []
with open('data/curriculos/2747150211073176/article_crossref.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        artigo = json.loads(line)
        artigos.append(artigo)
len(artigos)

258

# Authors

In [5]:
lattes_id = '2747150211073176'
author_db = get_or_create_profile(session, lattes_id)
author_db.given_name

In [26]:
artigos = []
with open('data/curriculos/2747150211073176/article_crossref.jsonl', "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        artigo = json.loads(line)
        artigos.append(artigo)
len(artigos)

258

In [5]:
authors = []
for artigo in artigos:
    authors.extend(artigo["author"])
len(authors)

1402

In [ ]:
stm = select(Author)
autores = session.execute(stm).scalars().all()
len(autores)

2026-04-19 10:16:09,412 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.normalized_full_name, authors.canonical_source, authors.needs_review, authors.affiliation_id 
FROM authors
2026-04-19 10:16:09,415 INFO sqlalchemy.engine.Engine [generated in 0.00243s] {}


1

In [ ]:
artigo = artigos[0]
publication = parser_publication(artigo)
container = parser_container(artigo)
funders = parse_funder(artigo)
references = parser_references(artigo)
metrics = extract_metrics(artigo)
authors = artigo["author"]

publication_db = get_or_create_publication(session, publication)
container_db = get_or_create_container(session, container)
publication_db.container = container_db
for author in authors:
    parsed_author, affiliation = parser_author_crossref(author)
    contributor = parser_contributor(author, parsed_author)
    author_db, created = get_or_create_author(session, parsed_author)
    if affiliation and not author_db.affiliation:
        author_db = affiliation_to_author(session, author_db, affiliation)
    link = authors_to_publication(session, publication_db, author_db, contributor )
publication_db = ingest_publication_funders(session, publication_db, funders)
publication_db = replace_publication_references(session, publication_db, references)

In [16]:
from lib.db.models import PublicationMetric


snapshot = PublicationMetric(**metrics)

        

    

In [ ]:
for artigo in artigos:
    publication = parser_publication(artigo)
    container = parser_container(artigo)
    funders = parse_funder(artigo)
    
    publication_db = get_or_create_publication(session, publication)
    container_db = get_or_create_container(session, container)
    publication_db.container = container_db
    # authors
    authors = artigo["author"]
    for author in authors:
        parsed_author, affiliation = parser_author_crossref(author)
        contributor = parser_contributor(author, parsed_author)
        author_db, created = get_or_create_author(session, parsed_author)
        # affiliation
        if affiliation and not author_db.affiliation:
            author_db = affiliation_to_author(session, author_db, affiliation)
        link = authors_to_publication(session, publication_db, author_db, contributor )
        
    # funders
    if funders:
        publication_db = ingest_publication_funders(session, publication_db, funders)
        
    # references
    references = parser_references(artigo)
    if references:
        publication_db = replace_publication_references(session, publication_db, references)
    # metrics
    metrics = extract_metrics(artigo)
    metrics_db = PublicationMetric(**metrics)
    session.add(publication_db)
    session.commit()
    
    
    print("INJET --->>>>:",artigos.index(artigo))

In [19]:
session.commit()

In [ ]:
for author in authors:
  parsed_author, affiliation = parser_author_crossref(author)
  author_db, created = get_or_create_author(session, parsed_author)
  if affiliation and not author_db.affiliation:
    author_db = affiliation_to_author(session, author_db, affiliation)

  if created:
    print(f"AUTHOR CRIADO: {authors.index(author)}", author_db.full_name)
  else:
    print(f'NOME ENCONTRADO: {authors.index(author)}', author_db.full_name)

# Publications

In [ ]:
for artigo in artigos:
    publication = parser_publication(artigo)
    publication_db = get_or_create_publication(session, publication)
    container = parser_container(artigo)
    container_db = get_or_create_container(session, container)
    publication_db.container = container_db
    session.flush()
    print("PUBLI:",publication_db.title )

In [ ]:
for artigo in artigos:
    publication = parser_publication(artigo)
    publication_db = get_or_create_publication(session, publication)
    authors = artigo["author"]
    for author in authors:
        parsed_author, affiliation = parser_author_crossref(author)
        contributor = parser_contributor(author, parsed_author)
        author_db, created = get_or_create_author(session, parsed_author)
        link = authors_to_publication(session, publication_db, author_db, contributor )
        print(link.id)

In [ ]:
artigo = artigos[0]
publication = parser_publication(artigo)
publication_db = get_or_create_publication(session, publication)
authors = artigo["author"]
for author in authors:
    parsed_author, affiliation = parser_author_crossref(author)
    contributor = parser_contributor(author, parsed_author)
    author_db, created = get_or_create_author(session, parsed_author)
    link = authors_to_publication(session, publication_db, author_db, contributor )
    print(link.id)



In [15]:
from sqlalchemy import select, func, desc
# from app.models import Author, PublicationContributor

In [17]:
stmt = (
    select(
        Author.id,
        Author.full_name,
        func.count(func.distinct(PublicationContributor.publication_id)).label("total_publications")
    )
    .join(PublicationContributor, PublicationContributor.author_id == Author.id)
    .group_by(Author.id, Author.full_name)
    .order_by(desc("total_publications"))
    .limit(10)
)

result = session.execute(stmt).all()

2026-04-15 22:02:32,160 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, count(distinct(publication_contributors.publication_id)) AS total_publications 
FROM authors INNER JOIN publication_contributors ON publication_contributors.author_id = authors.id GROUP BY authors.id, authors.full_name ORDER BY total_publications DESC 
 LIMIT %(param_1)s
2026-04-15 22:02:32,161 INFO sqlalchemy.engine.Engine [generated in 0.00123s] {'param_1': 10}


In [9]:
stm = select(Author).where(Author.needs_review)
result = session.execute(stm).scalars().all()
len(result)

2026-04-16 14:16:29,541 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.normalized_full_name, authors.canonical_source, authors.needs_review, authors.affiliation_id 
FROM authors 
WHERE authors.needs_review = 1
2026-04-16 14:16:29,543 INFO sqlalchemy.engine.Engine [generated in 0.00203s] {}


181

# Funders

In [ ]:
for artigo in artigos:
    publication = parser_publication(artigo)
    funders = parse_funder(artigo)
    if funders:
        publication_db = get_or_create_publication(session, publication)
        publication_db = ingest_publication_funders(session, publication_db, funders)
        print("PUBLIC ATUALIZADA:", publication_db.title)

# References

In [31]:
artigo = artigos[0]
references = parser_references(artigo)
reference = references[0]
reference

{'doi': '10.1016/j.ijpara.2020.04.007',
 'title': 'Why ignoring parasites in fish ecology is a mistake',
 'author': 'Timi',
 'journal_title': 'Int. J. Parasitol.',
 'year': '2020',
 'volume': '50',
 'issue': None,
 'match_source': None}

In [33]:
len(references)

61

In [ ]:
publi_ref = []
for artigo in artigos:
    references = parser_references(artigo)
    if references:
        publication = parser_publication(artigo)
        publication_db = get_or_create_publication(session, publication)
        publication_db = replace_publication_references(session, publication_db, references)
        publi_ref.append(publication_db)

In [28]:
publi = publi_ref[0]
len(publi.outgoing_references)

61

In [29]:
publi.title

'Immunometabolic costs of parasitism under warming: Impaired mitochondrial function and thermal tolerance in an Amazonian fish'

# Metrics

In [8]:
from lib.parser.metrics import extract_metric_snapshot
from lib.db.models import PublicationMetricSnapshot

In [ ]:
artigo = artigos[0]
publication = parser_publication(artigo)
publication_db = get_or_create_publication(session, publication)
metrics = extract_metric_snapshot(artigo)


2026-04-19 09:49:35,228 INFO sqlalchemy.engine.Engine INSERT INTO publication_metric_snapshots (publication_id, citation_count, reference_count, altmetric_score, mendeley_readers, tweets_count, news_count, blog_count, policy_count, patent_count, source) VALUES (%(publication_id)s, %(citation_count)s, %(reference_count)s, %(altmetric_score)s, %(mendeley_readers)s, %(tweets_count)s, %(news_count)s, %(blog_count)s, %(policy_count)s, %(patent_count)s, %(source)s) RETURNING publication_metric_snapshots.id, publication_metric_snapshots.collected_at
2026-04-19 09:49:35,231 INFO sqlalchemy.engine.Engine [cached since 159.4s ago] {'publication_id': 1, 'citation_count': 0, 'reference_count': 63, 'altmetric_score': None, 'mendeley_readers': None, 'tweets_count': None, 'news_count': None, 'blog_count': None, 'policy_count': None, 'patent_count': None, 'source': 'crossref'}
2026-04-19 09:49:35,252 INFO sqlalchemy.engine.Engine COMMIT


In [18]:
last_snapshot = publication_db.metric_snapshots[-1]
last_snapshot

In [20]:
last_snapshot.json()

AttributeError: 'PublicationMetricSnapshot' object has no attribute 'json'

In [ ]:
for artigo in artigos:
    publication = parser_publication(artigo)
    metrics = extract_metric_snapshot(artigo)
    publication_db = get_or_create_publication(session, publication)
    publication_db.metric_snapshots.append(snapshot)
    print(metrics)

{'citation_count': 0, 'reference_count': 63, 'altmetric_score': None, 'mendeley_readers': None, 'tweets_count': None, 'news_count': None, 'blog_count': None, 'policy_count': None, 'patent_count': None, 'source': 'crossref'}
{'citation_count': 0, 'reference_count': 12, 'altmetric_score': None, 'mendeley_readers': None, 'tweets_count': None, 'news_count': None, 'blog_count': None, 'policy_count': None, 'patent_count': None, 'source': 'crossref'}
{'citation_count': 0, 'reference_count': 104, 'altmetric_score': None, 'mendeley_readers': None, 'tweets_count': None, 'news_count': None, 'blog_count': None, 'policy_count': None, 'patent_count': None, 'source': 'crossref'}
{'citation_count': 1, 'reference_count': 54, 'altmetric_score': None, 'mendeley_readers': None, 'tweets_count': None, 'news_count': None, 'blog_count': None, 'policy_count': None, 'patent_count': None, 'source': 'crossref'}
{'citation_count': 0, 'reference_count': 57, 'altmetric_score': None, 'mendeley_readers': None, 'tweets

# Similaridade

In [13]:
from sqlalchemy import select, and_

In [154]:
session.commit()

2026-04-16 16:57:48,187 INFO sqlalchemy.engine.Engine COMMIT


In [155]:
needs_review = session.scalars(select(Author).where(Author.needs_review)).all()
len(needs_review)

2026-04-16 16:57:50,588 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-16 16:57:50,593 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.normalized_full_name, authors.canonical_source, authors.needs_review, authors.affiliation_id 
FROM authors 
WHERE authors.needs_review = 1
2026-04-16 16:57:50,594 INFO sqlalchemy.engine.Engine [cached since 7824s ago] {}


178

In [151]:
def get_candidates(author):
    reviews = []
    candidates = session.scalars(select(Author).where(Author.id != author.id)).all()
    for candidate in candidates:
        res = check_similarity(author.normalized_full_name, candidate.normalized_full_name)
        verdict = res.get("verdict")
        if verdict == 'duplicate' or verdict == 'review':
            reviews.append(candidate)
    return reviews
            
reviews = get_candidates(review)  

2026-04-16 16:57:18,715 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.normalized_full_name, authors.canonical_source, authors.needs_review, authors.affiliation_id 
FROM authors 
WHERE authors.id != %(id_1)s
2026-04-16 16:57:18,718 INFO sqlalchemy.engine.Engine [cached since 676.6s ago] {'id_1': 3}


In [152]:
reviews

[]

In [119]:
review.needs_review = False
session.add(review)
session.commit()

2026-04-16 16:38:13,505 INFO sqlalchemy.engine.Engine UPDATE authors SET needs_review=%(needs_review)s WHERE authors.id = %(authors_id)s
2026-04-16 16:38:13,507 INFO sqlalchemy.engine.Engine [generated in 0.00273s] {'needs_review': 0, 'authors_id': 1}
2026-04-16 16:38:13,515 INFO sqlalchemy.engine.Engine COMMIT


In [145]:
duplicate = reviews[0]
duplicate.normalized_full_name

'waldir d. heinrichs-caldas'

In [144]:
review.normalized_full_name

'waldir heinrichs-caldas'

In [146]:
def merge_duplicate(session, canonical, duplicate):
    stm1 = select(Author).where(Author.normalized_full_name == canonical)
    cano = session.scalars(stm1).first()
    stm1 = select(Author).where(Author.normalized_full_name == duplicate)
    dupl = session.scalars(stm1).first()
    for authorship in dupl.authorships:
        authorship.author_id = cano.id
        session.add(authorship)
        session.commit()
        
    if len(dupl.authorships) == 0:
        print("Todos os relaciomanetos escluidos")
        session.delete(dupl)
        session.flush()
        session.commit()
    else: 
        print('ainda falata')
    
    return cano, dupl

canonical = review.normalized_full_name
duplicate = duplicate.normalized_full_name 
cano, dupl = merge_duplicate(session, canonical, duplicate)

2026-04-16 16:56:04,338 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.normalized_full_name, authors.canonical_source, authors.needs_review, authors.affiliation_id 
FROM authors 
WHERE authors.normalized_full_name = %(normalized_full_name_1)s
2026-04-16 16:56:04,342 INFO sqlalchemy.engine.Engine [cached since 1844s ago] {'normalized_full_name_1': 'waldir heinrichs-caldas'}
2026-04-16 16:56:04,353 INFO sqlalchemy.engine.Engine SELECT authors.id, authors.full_name, authors.given_name, authors.family_name, authors.orcid, authors.lattes_id, authors.is_inpa_researcher, authors.normalized_full_name, authors.canonical_source, authors.needs_review, authors.affiliation_id 
FROM authors 
WHERE authors.normalized_full_name = %(normalized_full_name_1)s
2026-04-16 16:56:04,355 INFO sqlalchemy.engine.Engine [cached since 1844s ago] {'normalized_full_name_1': 'waldir d. 

In [133]:
for r in reviews:
    print(r.normalized_full_name)

waldir d. heinrichs-caldas


In [109]:
len(cano.authorships)

2026-04-16 16:33:47,507 INFO sqlalchemy.engine.Engine SELECT publication_contributors.id AS publication_contributors_id, publication_contributors.publication_id AS publication_contributors_publication_id, publication_contributors.author_id AS publication_contributors_author_id, publication_contributors.role AS publication_contributors_role, publication_contributors.`position` AS publication_contributors_position, publication_contributors.raw_name AS publication_contributors_raw_name, publication_contributors.raw_affiliation AS publication_contributors_raw_affiliation 
FROM publication_contributors 
WHERE %(param_1)s = publication_contributors.author_id
2026-04-16 16:33:47,508 INFO sqlalchemy.engine.Engine [cached since 2661s ago] {'param_1': 1}


245

In [112]:
len(dupl.authorships)

0

In [113]:
dupl

In [102]:
for authorship in dupl.authorships:
    authorship.author_id = cano.id
    session.add(authorship)
    session.flush()
    session.commit()
    print(authorship.author_id)

2026-04-16 16:30:38,141 INFO sqlalchemy.engine.Engine UPDATE publication_contributors SET author_id=%(author_id)s WHERE publication_contributors.id = %(publication_contributors_id)s
2026-04-16 16:30:38,142 INFO sqlalchemy.engine.Engine [cached since 1325s ago] {'author_id': 1, 'publication_contributors_id': 1748}
2026-04-16 16:30:38,146 INFO sqlalchemy.engine.Engine COMMIT
2026-04-16 16:30:38,155 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-16 16:30:38,157 INFO sqlalchemy.engine.Engine SELECT publication_contributors.id AS publication_contributors_id, publication_contributors.publication_id AS publication_contributors_publication_id, publication_contributors.author_id AS publication_contributors_author_id, publication_contributors.role AS publication_contributors_role, publication_contributors.`position` AS publication_contributors_position, publication_contributors.raw_name AS publication_contributors_raw_name, publication_contributors.raw_affiliation AS publication_contribu

In [101]:
cano.id

1

In [79]:
len(dup.authorships)

2026-04-16 16:12:13,365 INFO sqlalchemy.engine.Engine SELECT authors.id AS authors_id, authors.full_name AS authors_full_name, authors.given_name AS authors_given_name, authors.family_name AS authors_family_name, authors.orcid AS authors_orcid, authors.lattes_id AS authors_lattes_id, authors.is_inpa_researcher AS authors_is_inpa_researcher, authors.normalized_full_name AS authors_normalized_full_name, authors.canonical_source AS authors_canonical_source, authors.needs_review AS authors_needs_review, authors.affiliation_id AS authors_affiliation_id 
FROM authors 
WHERE authors.id = %(pk_1)s
2026-04-16 16:12:13,366 INFO sqlalchemy.engine.Engine [cached since 53.27s ago] {'pk_1': 135}
2026-04-16 16:12:13,370 INFO sqlalchemy.engine.Engine SELECT publication_contributors.id AS publication_contributors_id, publication_contributors.publication_id AS publication_contributors_publication_id, publication_contributors.author_id AS publication_contributors_author_id, publication_contributors.role 

0

In [105]:
dupl.id

176

In [106]:
session.delete(dupl)
session.flush()
session.commit()

2026-04-16 16:31:45,902 INFO sqlalchemy.engine.Engine SELECT lattes.id AS lattes_id_1, lattes.author_id AS lattes_author_id, lattes.lattes_id AS lattes_lattes_id, lattes.lattes_update AS lattes_lattes_update, lattes.html AS lattes_html, lattes.updated_at AS lattes_updated_at 
FROM lattes 
WHERE %(param_1)s = lattes.author_id
2026-04-16 16:31:45,905 INFO sqlalchemy.engine.Engine [cached since 1002s ago] {'param_1': 176}
2026-04-16 16:31:45,909 INFO sqlalchemy.engine.Engine DELETE FROM authors WHERE authors.id = %(id)s
2026-04-16 16:31:45,910 INFO sqlalchemy.engine.Engine [cached since 1002s ago] {'id': 176}
2026-04-16 16:31:45,914 INFO sqlalchemy.engine.Engine COMMIT


In [88]:
session.commit()

2026-04-16 16:14:56,226 INFO sqlalchemy.engine.Engine COMMIT


In [48]:
dup_contributor = dup.authorships[0]
dup_contributor.author_id

135

In [52]:
cano.full_name

'Adalberto Luis Val'

In [53]:
dup_contributor.author = cano
session.add(dup_contributor)
session.flush()

2026-04-16 16:06:34,118 INFO sqlalchemy.engine.Engine UPDATE publication_contributors SET author_id=%(author_id)s WHERE publication_contributors.id = %(publication_contributors_id)s
2026-04-16 16:06:34,123 INFO sqlalchemy.engine.Engine [generated in 0.00529s] {'author_id': 1, 'publication_contributors_id': 1615}


In [80]:
len(cano.authorships)

2026-04-16 16:12:27,855 INFO sqlalchemy.engine.Engine SELECT publication_contributors.id AS publication_contributors_id, publication_contributors.publication_id AS publication_contributors_publication_id, publication_contributors.author_id AS publication_contributors_author_id, publication_contributors.role AS publication_contributors_role, publication_contributors.`position` AS publication_contributors_position, publication_contributors.raw_name AS publication_contributors_raw_name, publication_contributors.raw_affiliation AS publication_contributors_raw_affiliation 
FROM publication_contributors 
WHERE %(param_1)s = publication_contributors.author_id
2026-04-16 16:12:27,857 INFO sqlalchemy.engine.Engine [cached since 1488s ago] {'param_1': 1}


240